In [3]:
import numpy as np
import pandas as pd

# ==============================================================================
# CELL 1: DATA GENERATION (Pure NumPy)
# ==============================================================================
# Set seed for reproducible results
np.random.seed(42)

regions = [
    "Addis Ababa", "Oromia", "Amhara", "Sidama", "SNNP",
    "Tigray", "Dire Dawa", "Harari", "Afar", "Benishangul-Gumuz"
]
n_regions = len(regions)

# Simulate enrollment numbers using random integer ranges
male_enrolled = np.random.randint(12000, 95000, size=n_regions)
female_enrolled = np.random.randint(10000, 90000, size=n_regions)

# Simulate exam pass rates (ranging from ~4% to 18%)
male_pass_rate = np.random.uniform(0.04, 0.18, size=n_regions)
# Introduce gender disparity in pass rates
female_pass_rate = male_pass_rate * np.random.uniform(0.85, 0.98, size=n_regions)

# Calculate counts of students who passed
male_passed = (male_enrolled * male_pass_rate).astype(int)
female_passed = (female_enrolled * female_pass_rate).astype(int)

# Create raw Pandas DataFrame
df = pd.DataFrame({
    "Region": regions,
    "Male_Enrolled": male_enrolled,
    "Female_Enrolled": female_enrolled,
    "Male_Passed": male_passed,
    "Female_Passed": female_passed
})

# ==============================================================================
# CELL 2: FEATURE ENGINEERING & METRIC CALCULATIONS (Pandas Vectorization)
# ==============================================================================
# 1. Total student counts
df["Total_Enrolled"] = df["Male_Enrolled"] + df["Female_Enrolled"]
df["Total_Passed"] = df["Male_Passed"] + df["Female_Passed"]

# 2. Percentage Pass Rates
df["Overall_Pass_Rate_%"] = np.round((df["Total_Passed"] / df["Total_Enrolled"]) * 100, 2)
df["Male_Pass_Rate_%"] = np.round((df["Male_Passed"] / df["Male_Enrolled"]) * 100, 2)
df["Female_Pass_Rate_%"] = np.round((df["Female_Passed"] / df["Female_Enrolled"]) * 100, 2)

# 3. Gender Parity Indices (GPI)
# (Ratio = Female / Male. Value of 1.0 = equal performance/enrollment)
df["Enrollment_GPI"] = np.round(df["Female_Enrolled"] / df["Male_Enrolled"], 2)
df["Pass_Rate_GPI"] = np.round(df["Female_Pass_Rate_%"] / df["Male_Pass_Rate_%"], 2)

# 4. Categorize Parity Status using NumPy conditions
conditions = [
    (df["Pass_Rate_GPI"] >= 0.95),
    (df["Pass_Rate_GPI"] >= 0.90) & (df["Pass_Rate_GPI"] < 0.95),
    (df["Pass_Rate_GPI"] < 0.90)
]
choices = ["Near Parity", "Moderate Male Advantage", "High Male Advantage"]
df["Parity_Status"] = np.select(conditions, choices, default="Unknown")

# 5. Bin regions by size using Pandas Quantiles
df["Region_Scale"] = pd.qcut(
    df["Total_Enrolled"],
    q=3,
    labels=["Small Region", "Medium Region", "Large Region"]
)

# ==============================================================================
# CELL 3: NATIONAL AGGREGATION ROW
# ==============================================================================
national_total_enrolled = df["Total_Enrolled"].sum()
national_total_passed = df["Total_Passed"].sum()
national_male_enrolled = df["Male_Enrolled"].sum()
national_female_enrolled = df["Female_Enrolled"].sum()
national_male_passed = df["Male_Passed"].sum()
national_female_passed = df["Female_Passed"].sum()

national_row = pd.DataFrame([{
    "Region": "NATIONAL TOTAL",
    "Male_Enrolled": national_male_enrolled,
    "Female_Enrolled": national_female_enrolled,
    "Male_Passed": national_male_passed,
    "Female_Passed": national_female_passed,
    "Total_Enrolled": national_total_enrolled,
    "Total_Passed": national_total_passed,
    "Overall_Pass_Rate_%": round((national_total_passed / national_total_enrolled) * 100, 2),
    "Male_Pass_Rate_%": round((national_male_passed / national_male_enrolled) * 100, 2),
    "Female_Pass_Rate_%": round((national_female_passed / national_female_enrolled) * 100, 2),
    "Enrollment_GPI": round(national_female_enrolled / national_male_enrolled, 2),
    "Pass_Rate_GPI": round((national_female_passed / national_female_enrolled) / (national_male_passed / national_male_enrolled), 2),
    "Parity_Status": "National Baseline",
    "Region_Scale": "All Regions"
}])

# Combine regional table with national totals
full_table = pd.concat([df, national_row], ignore_index=True)

# ==============================================================================
# CELL 4: INSIGHTS & JUPYTER VISUAL STYLING
# ==============================================================================
print("=" * 65)
print("              TOP 3 HIGHEST PERFORMING REGIONS")
print("=" * 65)
top_3 = df.sort_values(by="Overall_Pass_Rate_%", ascending=False)[
    ["Region", "Total_Enrolled", "Overall_Pass_Rate_%", "Pass_Rate_GPI"]
].head(3)
print(top_3.to_string(index=False))

print("\n" + "=" * 65)
print("          AVERAGE PERFORMANCE BY REGIONAL SCALE")
print("=" * 65)
scale_summary = df.groupby("Region_Scale", observed=False)[
    ["Total_Enrolled", "Overall_Pass_Rate_%", "Pass_Rate_GPI"]
].mean().round(2)
print(scale_summary.to_string())
print("\n")

# Render styled interactive HTML table directly in Jupyter Notebook
full_table.style.background_gradient(
    cmap="YlGn", subset=["Overall_Pass_Rate_%", "Male_Pass_Rate_%", "Female_Pass_Rate_%"]
).background_gradient(
    cmap="Reds_r", subset=["Pass_Rate_GPI"]
)

              TOP 3 HIGHEST PERFORMING REGIONS
           Region  Total_Enrolled  Overall_Pass_Rate_%  Pass_Rate_GPI
        Dire Dawa          124119                16.05           0.85
Benishangul-Gumuz           91730                11.78           0.90
      Addis Ababa           78885                11.74           0.90

          AVERAGE PERFORMANCE BY REGIONAL SCALE
               Total_Enrolled  Overall_Pass_Rate_%  Pass_Rate_GPI
Region_Scale                                                     
Small Region         83028.50                 9.34           0.93
Medium Region       101976.67                10.62           0.91
Large Region        155027.00                 5.25           0.93




,Region,Male_Enrolled,Female_Enrolled,Male_Passed,Female_Passed,Total_Enrolled,Total_Passed,Overall_Pass_Rate_%,Male_Pass_Rate_%,Female_Pass_Rate_%,Enrollment_GPI,Pass_Rate_GPI,Parity_Status,Region_Scale
0,Addis Ababa,27795,51090,3491,5774,78885,9265,11.740000,12.560000,11.300000,1.840000,0.900000,Moderate Male Advantage,Small Region
1,Oromia,12860,77221,527,3095,90081,3622,4.020000,4.100000,4.010000,6.000000,0.980000,Near Parity,Medium Region
2,Amhara,88820,74820,3839,2945,163640,6784,4.150000,4.320000,3.940000,0.840000,0.910000,Moderate Male Advantage,Large Region
3,Sidama,66886,10769,7589,1175,77655,8764,11.290000,11.350000,10.910000,0.160000,0.960000,Near Parity,Small Region
4,SNNP,18265,69735,1753,6281,88000,8034,9.130000,9.600000,9.010000,3.820000,0.940000,Moderate Male Advantage,Small Region
5,Tigray,94386,72955,4392,3084,167341,7476,4.470000,4.650000,4.230000,0.770000,0.910000,Moderate Male Advantage,Large Region
6,Dire Dawa,49194,74925,8674,11252,124119,19926,16.050000,17.630000,15.020000,1.520000,0.850000,High Male Advantage,Medium Region
7,Harari,56131,77969,4074,5503,134100,9577,7.140000,7.260000,7.060000,1.390000,0.970000,Near Parity,Large Region
8,Afar,72263,15311,3807,744,87574,4551,5.200000,5.270000,4.860000,0.210000,0.920000,Moderate Male Advantage,Small Region
9,Benishangul-Gumuz,28023,63707,3546,7258,91730,10804,11.780000,12.650000,11.390000,2.270000,0.900000,Moderate Male Advantage,Medium Region
